In [45]:
!pip install -q pymongo azure-storage-file-datalake

#Importacoes
import hashlib
import json
import re
import time
import unicodedata
import warnings
from collections import Counter
from datetime import datetime, timedelta, timezone
from pprint import pprint

from azure.storage.filedatalake import FileSystemClient
from pymongo import MongoClient, ReplaceOne
from pymongo.errors import BulkWriteError, OperationFailure

# O pymongo avisa que está falando com um Cosmos DB
warnings.filterwarnings("ignore", message="You appear to be connected to a CosmosDB cluster")


In [46]:
!curl -s ifconfig.me

34.135.65.67

In [47]:
def ler_segredo(nome):
    #acessa o painel de secrets do collab para pegar as strings de conexoes com o datalake e mongo
    try:
        from google.colab import userdata
        valor = userdata.get(nome)
        if valor:
            return valor
    except Exception:
        pass
    from getpass import getpass
    #caso nao ache nada, solicita a chave
    return getpass(f"Informe {nome}: ")


COSMOS_CONNECTION_STRING = ler_segredo("COSMOS_CONNECTION_STRING")
DATALAKE_SAS_TOKEN = ler_segredo("DATALAKE_SAS_TOKEN")

print("Credenciais carregadas")

Credenciais carregadas


In [74]:
#salva o nome do banco mongo
NOME_BANCO = "conectabeauty"

#salva o nome das colecoes
NOME_COLECAO_PEDIDOS = "pedidos_canais_externos"
NOME_COLECAO_LOGS = "logs_tecnicos" # e populada automaticamente pela propria aplicacao

# salva o caminho de onde vem o arquivo para ser tratado para ser inserido no mongo
DATALAKE_ACCOUNT_URL = "https://datalakeconectabeauty.dfs.core.windows.net"
DATALAKE_CONTAINER = "bronze"
DATALAKE_PASTA = "gerados_por_script"

# delimita quantos documentos sao processados de uma vez
LIMITE_DOCUMENTOS = None

# delimita a quantidade de documentos inserido no banco de uma vez com insert_many para não explodir o RU do cosmos DB
TAMANHO_LOTE = 100

# opcao de recriar a colecao do zero, apagando tudo e inserindo novamente
RECRIAR_COLECAO = True

# Rotinas automatizadas.
LIMITE_CONFERENCIA = 30000  # pedidos acima deste valor (R$) são marcados para conferência
DIAS_RETENCAO_LOGS = 30     # requisições mais antigas que isso são removidas dos logs técnicos
REPETICOES_ROTINAS = 1      # quantas vezes o agendador da seção 11 executa as rotinas
INTERVALO_ROTINAS_S = 60    # intervalo entre as execuções, quando REPETICOES_ROTINAS > 1

In [49]:
#salva a conexao em um string
client = MongoClient(COSMOS_CONNECTION_STRING, serverSelectionTimeoutMS=10000)

#verifica se a conexao foi estabelecida
try:
    client.admin.command("ping")
except Exception as erro:
    raise ConnectionError(f"Não foi possível conectar ao Azure Cosmos DB: {erro}") from erro

db = client[NOME_BANCO]

print("Conexão com o Azure Cosmos DB realizada com sucesso!")
print(f"Banco: {NOME_BANCO}")
print(f"Coleções existentes: {db.list_collection_names()}")

Conexão com o Azure Cosmos DB realizada com sucesso!
Banco: conectabeauty
Coleções existentes: ['pedidos_canais_externos', 'logs_tecnicos']


In [50]:
#verifica se vai precisar recriar a colecao de pedidos
if RECRIAR_COLECAO and NOME_COLECAO_PEDIDOS in db.list_collection_names():
    db.drop_collection(NOME_COLECAO_PEDIDOS)
    print(f"Coleção '{NOME_COLECAO_PEDIDOS}' apagada para recriação")

#verifica se ja existe a colecao e se nao existir cria
if NOME_COLECAO_PEDIDOS in db.list_collection_names():
    print(f"Coleção '{NOME_COLECAO_PEDIDOS}' já existe")
else:
    db.create_collection(NOME_COLECAO_PEDIDOS)
    print(f"Coleção '{NOME_COLECAO_PEDIDOS}' criada")

pedidos_col = db[NOME_COLECAO_PEDIDOS]
logs_col = db[NOME_COLECAO_LOGS]

# cada pedido e gravado pelo _id, se a coleção ja foi criada antes com outra chave de partição (shard key), a gravacao nao vai funcionar
try:
    #procura pela colecao e verifica a shard key dela
    info = db.command({"customAction": "GetCollection", "collection": NOME_COLECAO_PEDIDOS})
    chave = info.get("shardKeyDefinition")
    print(f"Chave de partição: {chave or 'nenhuma (coleção não particionada)'}")

    # se a chave for diferente de _id, obriga a recriar a colecao
    if chave and list(chave) != ["_id"]:
        raise RuntimeError(
            f"A coleção foi criada com a chave de partição {chave}. "
            "Defina RECRIAR_COLECAO = True na seção 2 e rode novamente."
        )
except OperationFailure:
    print("Metadados de partição indisponíveis (comando exclusivo do Cosmos DB) — seguindo.")

#define os indices da  colecao e cria eles
INDICES = [
    "padronizado.canal",
    "padronizado.valor_total",
    "padronizado.data_pedido",
    "padronizado.forma_pagamento",
    "padronizado.estado",
]
for campo in INDICES:
    pedidos_col.create_index(campo)

print(f"Índices da coleção: {sorted(pedidos_col.index_information())}")

Coleção 'pedidos_canais_externos' apagada para recriação
Coleção 'pedidos_canais_externos' criada
Chave de partição: nenhuma (coleção não particionada)
Índices da coleção: ['_id_', 'padronizado.canal_1', 'padronizado.data_pedido_1', 'padronizado.estado_1', 'padronizado.forma_pagamento_1', 'padronizado.valor_total_1']


In [51]:
#acessa o datalake
fs = FileSystemClient(
    account_url=DATALAKE_ACCOUNT_URL,
    file_system_name=DATALAKE_CONTAINER,
    credential=DATALAKE_SAS_TOKEN,
)

#encontra todos os arquivos dentro do caminho inserido anteriormente
arquivos = [
    caminho
    for caminho in fs.get_paths(path=DATALAKE_PASTA, recursive=True)
    if not caminho.is_directory and caminho.name.endswith(".jsonl")
]

if not arquivos:
    raise FileNotFoundError(f"Nenhum arquivo .jsonl encontrado em {DATALAKE_CONTAINER}/{DATALAKE_PASTA}.")


print(f"Arquivos .jsonl encontrados em {DATALAKE_CONTAINER}/{DATALAKE_PASTA}: {len(arquivos)}")
for caminho in arquivos:
    print(f"{caminho.name} ({caminho.content_length / 1024:,.1f} KB, modificado em {caminho.last_modified})")

Arquivos .jsonl encontrados em bronze/gerados_por_script: 1
gerados_por_script/pedidos_canais_externos.jsonl (990.6 KB, modificado em 2026-11-04 23:25:20)


In [52]:
#lista que vai guardar todos os pedidos lidos
registros_brutos = []

#percorre cada arquivo .jsonl encontrado no data lake
for caminho in arquivos:
    #baixa o arquivo inteiro do data lake e transforma de bytes pra texto
    conteudo = fs.get_file_client(caminho.name).download_file().readall().decode("utf-8")

    #contadores pra saber quantas linhas deram certo e quantas foram descartadas nesse arquivo
    linhas_validas = linhas_invalidas = 0

    #le linha por linha
    for numero_linha, linha in enumerate(conteudo.splitlines(), start=1):
        #pula as linhas vazias
        if not linha.strip():
            continue
        try:
            #converte a linha pra dicionario e guarda com a origem, pra conseguir rastrear depois
            registros_brutos.append({"arquivo": caminho.name, "linha": numero_linha, "doc": json.loads(linha)})
            linhas_validas += 1
        except json.JSONDecodeError:
            #se a linha nao for um json valido, descarta e so conta
            linhas_invalidas += 1

    print(f"{caminho.name}: {linhas_validas} pedidos lidos, {linhas_invalidas} linhas inválidas descartadas")

#se tiver um limite definido, usa so os primeiros pedidos
if LIMITE_DOCUMENTOS:
    registros_brutos = registros_brutos[:LIMITE_DOCUMENTOS]
    print(f"\nLIMITE_DOCUMENTOS ativo: usando apenas os primeiros {len(registros_brutos)} pedidos.")

print(f"\nTotal de pedidos brutos carregados: {len(registros_brutos)}")

gerados_por_script/pedidos_canais_externos.jsonl: 2000 pedidos lidos, 0 linhas inválidas descartadas

Total de pedidos brutos carregados: 2000


In [53]:
#define o mapeamento dos canais, com o campo + valor e caso esteja none, nao e obrigado a ter aquele campo nos pedidos daquele canal
MAPA_CANAIS = {
    "Instagram": {
        "id": "pedido_instagram_id", "cliente": "cliente.nome", "telefone": "cliente.telefone",
        "cidade": "cliente.cidade", "estado": None, "documento": None, "itens": "itens",
        "valor": "valor_total", "frete": None, "pagamento": "forma_pagamento", "data": "data_pedido",
    },
    "Facebook": {
        "id": "id_pedido_fb", "cliente": "comprador_nome", "telefone": "comprador_telefone",
        "cidade": "cidade_entrega", "estado": None, "documento": "cpf_informado", "itens": "produtos",
        "valor": "total", "frete": None, "pagamento": "pagamento", "data": "criado_em",
    },
    "Marketplace": {
        "id": "order_id", "cliente": "comprador.nome", "telefone": "comprador.telefone",
        "cidade": "endereco_entrega.cidade", "estado": "endereco_entrega.estado",
        "documento": "comprador.documento", "itens": "itens_pedido", "valor": "valor_total_pedido",
        "frete": "valor_frete", "pagamento": "forma_pagamento", "data": "timestamp_pedido",
    },
    "Mercado Livre": {
        "id": "ml_order_id", "cliente": "buyer.full_name", "telefone": "buyer.phone",
        "cidade": "shipping.city", "estado": "shipping.state", "documento": "buyer.document",
        "itens": "items", "valor": "total_amount", "frete": "shipping.cost",
        "pagamento": "payment_method", "data": "order_date",
    },
}

#identificacao de cada documento para saber de qual canal e
def identificar_canal(doc):
    for canal, mapa in MAPA_CANAIS.items():
        if mapa["id"] in doc:
            return canal
    return None

#busca um valor dentro do documento usando um caminho com pontos, tipo "comprador.endereco.cidade"
def ler_caminho(doc, caminho):
    if caminho is None:
        return None
    #comeca no inicio e vai descendo nivel por nivel
    atual = doc
    for parte in caminho.split("."):
      #confirmando que veio algo no campo e formatado como dicionario
        if not isinstance(atual, dict):
            return None
        atual = atual.get(parte)
    return atual


#apenas para provar que os canais realmente tem formatos diferentes
exemplo_por_canal = {}
for registro in registros_brutos:
    canal = identificar_canal(registro["doc"])
    if canal and canal not in exemplo_por_canal:
        exemplo_por_canal[canal] = registro["doc"]

for canal, doc in exemplo_por_canal.items():
    print(f"{canal} campos do documento original")
    print(list(doc.keys()))

Mercado Livre campos do documento original
['channel', 'ml_order_id', 'buyer', 'shipping', 'items', 'total_amount', 'payment_method', 'reputation_bonus_applied', 'order_date']
Facebook campos do documento original
['origem', 'id_pedido_fb', 'comprador_nome', 'comprador_telefone', 'cidade_entrega', 'produtos', 'total', 'pagamento', 'criado_em', 'cpf_informado']
Marketplace campos do documento original
['canal_venda', 'order_id', 'comprador', 'endereco_entrega', 'itens_pedido', 'valor_total_pedido', 'valor_frete', 'forma_pagamento', 'timestamp_pedido']
Instagram campos do documento original
['canal', 'pedido_instagram_id', 'cliente', 'itens', 'valor_total', 'forma_pagamento', 'data_pedido', 'combinado_via_direct']


In [54]:
CAMPOS_SO_DIGITOS = {"documento", "document", "cpf_informado", "telefone", "phone", "comprador_telefone"}

#tira espaço duplo, deixa só dígitos em documento/telefone
def normalizar_texto(texto):
    texto = unicodedata.normalize("NFC", texto)
    return re.sub(r"\s+", " ", texto).strip()

#limpa de forma recursiva em qualquer formato de dado
def limpar_recursivo(valor, chave=None):
    if isinstance(valor, dict):
        return {k: limpar_recursivo(v, k) for k, v in valor.items()}
    if isinstance(valor, list):
        return [limpar_recursivo(v, chave) for v in valor]
    if isinstance(valor, str):
        if chave in CAMPOS_SO_DIGITOS:
            return re.sub(r"\D", "", valor)
        return normalizar_texto(valor)
    return valor

#valida cpf
def cpf_valido(cpf):
    if len(cpf) != 11 or cpf == cpf[0] * 11:
        return False
    for posicao in (9, 10):
        soma = sum(int(cpf[i]) * (posicao + 1 - i) for i in range(posicao))
        if (soma * 10) % 11 % 10 != int(cpf[posicao]):
            return False
    return True

#valida cnpj
def cnpj_valido(cnpj):
    if len(cnpj) != 14 or cnpj == cnpj[0] * 14:
        return False
    pesos = [5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2]
    for posicao, pesos_posicao in ((12, pesos), (13, [6] + pesos)):
        resto = sum(int(cnpj[i]) * pesos_posicao[i] for i in range(posicao)) % 11
        if (0 if resto < 2 else 11 - resto) != int(cnpj[posicao]):
            return False
    return True

#formato valor financeiro para o padrao brasileiro
def moeda(valor):
    return "R$ " + f"{valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


print("Funções de limpeza, validação e formatação definidas.")

Funções de limpeza, validação e formatação definidas.


In [55]:
def padronizar(doc, canal):
    #define os canais (instagram, marketplace, etc..)
    mapa = MAPA_CANAIS[canal]

    #para cada dicionario dentro do canal procura os respectivos valores
    itens = ler_caminho(doc, mapa["itens"]) or []
    documento = ler_caminho(doc, mapa["documento"])
    valor = ler_caminho(doc, mapa["valor"])
    frete = ler_caminho(doc, mapa["frete"])

    #extrai a data e formata
    try:
        data_pedido = datetime.fromisoformat(ler_caminho(doc, mapa["data"]))
    except (TypeError, ValueError):
        data_pedido = None

    #se não tiver documento retorna None
    tipo_documento = documento_ok = None

    #verifica se é CPF ou CNPJ e valida eles
    if documento:
        tipo_documento = {11: "CPF", 14: "CNPJ"}.get(len(documento))
        documento_ok = cpf_valido(documento) if tipo_documento == "CPF" else cnpj_valido(documento)

    #padroniza todo o documento json, para os campos diferentes entre os canais serem os mesmos para inserir no banco
    padronizado = {
        "canal": canal,
        "id_pedido_canal": str(ler_caminho(doc, mapa["id"])),
        "cliente": ler_caminho(doc, mapa["cliente"]),
        "telefone": ler_caminho(doc, mapa["telefone"]),
        "cidade": ler_caminho(doc, mapa["cidade"]),
        "estado": ler_caminho(doc, mapa["estado"]),
        "documento": documento,
        "tipo_documento": tipo_documento,
        "documento_valido": documento_ok,
        "produtos": [item.get("produto") for item in itens],
        "qtd_itens": sum(int(item.get("quantidade") or 0) for item in itens),
        "valor_total": round(float(valor), 2) if valor is not None else None,
        "frete": round(float(frete), 2) if frete is not None else None,
        "forma_pagamento": ler_caminho(doc, mapa["pagamento"]),
        "data_pedido": data_pedido,
    }

    #apenas validações extras para caso algum campo esteja despadronizado
    alertas = []
    if not padronizado["valor_total"]:
        alertas.append("valor_total ausente ou zerado")
    if not itens:
        alertas.append("pedido sem itens")
    if data_pedido is None:
        alertas.append("data do pedido ausente ou inválida")
    if documento and not documento_ok:
        alertas.append("documento inválido")
    padronizado["alertas_qualidade"] = alertas

    #retorna o dicionario com todos os documentos dos canais padronizados
    return padronizado


def tratar_registro(registro):
    bruto = registro["doc"]
    #identifica o canal do documento
    canal = identificar_canal(bruto)
    if canal is None:
        return None

    #aplica a limpeza no dado bruto
    tratado = limpar_recursivo(bruto)
    #gera um hash unico para evitar deduplicação ou inserir duas vezes o mesmo documento
    #se o mesmo documento aparecer duas vezes, ele não é tratado como diferente e sim como igual, o mongo ate cria o _id, mas ele não sabe o conteudo dentro dele
    tratado["_id"] = hashlib.sha1(json.dumps(bruto, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
    #padroniza os documentos
    tratado["padronizado"] = padronizar(tratado, canal)

    #adiciona um dicionario de metadados para rastreabilidade
    tratado["ingestao"] = {
        "origem": f"{DATALAKE_CONTAINER}/{registro['arquivo']}",
        "linha": registro["linha"],
        "carregado_em": datetime.now(timezone.utc),
        "alterado_na_limpeza": limpar_recursivo(bruto) != bruto,
    }
    return tratado

In [56]:
pedidos_tratados = {}
rejeitados = 0

#trata todos os registros vindo do documentos
for registro in registros_brutos:
    tratado = tratar_registro(registro)
    #caso algum registro esteja vazio, adiciona mais um em rejeitados para não subir no banco
    if tratado is None:
        rejeitados += 1
        continue

    pedidos_tratados[tratado["_id"]] = tratado  # mesmo _id = mesmo pedido, fica um só

#cria uma lista com todos os pedidos prontos para serem inseridos
pedidos_tratados = list(pedidos_tratados.values())

duplicados = len(registros_brutos) - rejeitados - len(pedidos_tratados)

#validação global
print("RESUMO DO TRATAMENTO")
print(f"Pedidos brutos lidos:{len(registros_brutos)}")
print(f"Canal não identificado:{rejeitados}")
print(f"Duplicados removidos:{duplicados}")
print(f"Pedidos prontos para gravar:{len(pedidos_tratados)}")
print(f"Alterados pela limpeza:{sum(p['ingestao']['alterado_na_limpeza'] for p in pedidos_tratados)}")
print(f"Com alerta de qualidade:{sum(bool(p['padronizado']['alertas_qualidade']) for p in pedidos_tratados)}")

print("\nPedidos por canal:")
for canal, quantidade in Counter(p["padronizado"]["canal"] for p in pedidos_tratados).most_common():
    print(f"  {canal:<15} {quantidade}")

RESUMO DO TRATAMENTO
Pedidos brutos lidos:2000
Canal não identificado:0
Duplicados removidos:0
Pedidos prontos para gravar:2000
Alterados pela limpeza:2000
Com alerta de qualidade:0

Pedidos por canal:
  Mercado Livre   522
  Marketplace     508
  Facebook        492
  Instagram       478


In [57]:
#apenas a prova pratica de que os documentos foram tratados
exemplo = next(r for r in registros_brutos if identificar_canal(r["doc"]) == "Marketplace")
depois = tratar_registro(exemplo)

print("ANTES (como veio do Data Lake)")
pprint(exemplo["doc"]["comprador"])

print("DEPOIS (documento e telefone limpos)")
pprint(depois["comprador"])

print("BLOCO PADRONIZADO ADICIONADO")
pprint(depois["padronizado"])

ANTES (como veio do Data Lake)
{'documento': '739.812.406-69',
 'nome': 'Joana Câmara',
 'telefone': '+55 (071) 7715-1739',
 'tipo_documento': 'CPF'}
DEPOIS (documento e telefone limpos)
{'documento': '73981240669',
 'nome': 'Joana Câmara',
 'telefone': '5507177151739',
 'tipo_documento': 'CPF'}
BLOCO PADRONIZADO ADICIONADO
{'alertas_qualidade': [],
 'canal': 'Marketplace',
 'cidade': 'Castro de Guerra',
 'cliente': 'Joana Câmara',
 'data_pedido': datetime.datetime(2023, 3, 21, 10, 39, 38),
 'documento': '73981240669',
 'documento_valido': True,
 'estado': 'CE',
 'forma_pagamento': 'Pix',
 'frete': 35.72,
 'id_pedido_canal': 'MKT729061',
 'produtos': ['Borracha de Lavatório', 'Aparador de C ??', 'Maca 3 posi????es'],
 'qtd_itens': 5,
 'telefone': '5507177151739',
 'tipo_documento': 'CPF',
 'valor_total': 2319.06}


In [58]:
CODIGO_DUPLICADO = 11000  # "DuplicateKeyError" - o _id (hash do conteudo) ja existe na colecao
COD_LIMITE_VAZAO = 16500  # "TooManyRequests" do Cosmos DB


def gravar_em_lotes(colecao, documentos, tamanho_lote=TAMANHO_LOTE, max_tentativas=10):
    novos = ja_existiam = 0

    #processa em lotes, pra nao mandar os 10 mil documentos numa unica chamada
    for inicio in range(0, len(documentos), tamanho_lote):
        lote = documentos[inicio:inicio + tamanho_lote]

        #tenta inserir o lote, tratando duplicado e limite de vazao separado de um erro de verdade
        for tentativa in range(1, max_tentativas + 1):
            try:
                #ordered=False: continua inserindo o resto do lote mesmo se algum documento falhar
                resultado = colecao.insert_many(lote, ordered=False)
                novos += len(resultado.inserted_ids)
                break
            except BulkWriteError as erro:
                detalhes = erro.details
                novos += detalhes.get("nInserted", 0)  #quantos entraram de verdade nessa tentativa, mesmo com erro no meio

                erros = detalhes.get("writeErrors", [])
                duplicados = [e for e in erros if e.get("code") == CODIGO_DUPLICADO]
                throttle = [e for e in erros if e.get("code") == COD_LIMITE_VAZAO]

                #duplicado e o esperado: o _id e o hash do documento, entao pedido repetido (ou notebook rodado de novo) sempre cai aqui
                ja_existiam += len(duplicados)

                #se sobrou erro que nao seja duplicado nem limite de vazao, algo quebrou de verdade - estoura
                if len(duplicados) + len(throttle) != len(erros):
                    raise

                if not throttle:
                    break  #so tinha duplicado, nao precisa tentar de novo

                #reenvia so os documentos que falharam por limite de vazao
                lote = [lote[e["index"]] for e in throttle]
            except OperationFailure as erro:
                #esse except pega quando o lote inteiro falhou (nao cada documento individual)
                if erro.code != COD_LIMITE_VAZAO:
                    raise
            #espera cada vez mais entre uma tentativa e outra, ate 30s
            time.sleep(min(0.5 * 2 ** tentativa, 30))
        else:
            raise RuntimeError(f"Lote iniciado em {inicio} excedeu {max_tentativas} tentativas.")

        gravados = min(inicio + tamanho_lote, len(documentos))
        if gravados % 1000 == 0 or gravados == len(documentos):
            print(f"  {gravados}/{len(documentos)} pedidos processados")

    return novos, ja_existiam


inicio_carga = time.time()
novos, ja_existiam = gravar_em_lotes(pedidos_col, pedidos_tratados)


print(f"Pedidos novos inseridos:{novos}")
print(f"Pedidos que ja existiam:{ja_existiam}")
print(f"Tempo total:{time.time() - inicio_carga:.1f}s")
print(f"Documentos na colecao agora:{pedidos_col.count_documents({})}")

  1000/2000 pedidos processados
  2000/2000 pedidos processados
Pedidos novos inseridos:2000
Pedidos que ja existiam:0
Tempo total:141.3s
Documentos na colecao agora:2000


In [59]:
#verificando quais bancos existem
client.list_database_names()

#Consulta simples: total de documentos e um pedido completo.
print(f"Total de pedidos na coleção: {pedidos_col.count_documents({})}\n")

print("Um pedido qualquer, completo:")
pprint(pedidos_col.find_one())

Total de pedidos na coleção: 2000

Um pedido qualquer, completo:
{'_id': '1be2c3642e85a49a5b84af6a7cdd3594ed624acf',
 'buyer': {'document': '02479368556',
           'full_name': 'Maria Clara Moreira',
           'phone': '5501166499935'},
 'channel': 'Mercado Livre',
 'ingestao': {'alterado_na_limpeza': True,
              'carregado_em': datetime.datetime(2026, 11, 17, 1, 39, 37, 311000),
              'linha': 1,
              'origem': 'bronze/gerados_por_script/pedidos_canais_externos.jsonl'},
 'items': [{'preco_unitario': 628.87,
            'produto': 'Arm??rio Utilit??rio 2 Portas',
            'quantidade': 3}],
 'ml_order_id': 'ML6595722310',
 'order_date': '2021-05-26T09:03:51',
 'padronizado': {'alertas_qualidade': [],
                 'canal': 'Mercado Livre',
                 'cidade': 'Ribeiro',
                 'cliente': 'Maria Clara Moreira',
                 'data_pedido': datetime.datetime(2021, 5, 26, 9, 3, 51),
                 'documento': '02479368556',
        

In [60]:
# distinct():
pedidos_col.distinct("padronizado.forma_pagamento")

['Cartao de credito', 'Cartao de debito', 'Pix', 'Boleto']

In [61]:
#Iteração sem filtro:
for doc in pedidos_col.find().limit(5):
    print(doc)

{'_id': '1be2c3642e85a49a5b84af6a7cdd3594ed624acf', 'channel': 'Mercado Livre', 'ml_order_id': 'ML6595722310', 'buyer': {'full_name': 'Maria Clara Moreira', 'document': '02479368556', 'phone': '5501166499935'}, 'shipping': {'city': 'Ribeiro', 'state': 'PR', 'cost': 10.8}, 'items': [{'produto': 'Arm??rio Utilit??rio 2 Portas', 'quantidade': 3, 'preco_unitario': 628.87}], 'total_amount': 1886.61, 'payment_method': 'Cartao de credito', 'reputation_bonus_applied': False, 'order_date': '2021-05-26T09:03:51', 'padronizado': {'canal': 'Mercado Livre', 'id_pedido_canal': 'ML6595722310', 'cliente': 'Maria Clara Moreira', 'telefone': '5501166499935', 'cidade': 'Ribeiro', 'estado': 'PR', 'documento': '02479368556', 'tipo_documento': 'CPF', 'documento_valido': True, 'produtos': ['Arm??rio Utilit??rio 2 Portas'], 'qtd_itens': 3, 'valor_total': 1886.61, 'frete': 10.8, 'forma_pagamento': 'Cartao de credito', 'data_pedido': datetime.datetime(2021, 5, 26, 9, 3, 51), 'alertas_qualidade': []}, 'ingestao'

In [62]:
#Filtro simples por igualdade:
filtro = {"padronizado.canal": "Instagram"}
print(f"Pedidos do Instagram: {pedidos_col.count_documents(filtro)}\n")

projecao = {"_id": 0, "padronizado.cliente": 1, "padronizado.cidade": 1, "padronizado.valor_total": 1}
for pedido in pedidos_col.find(filtro, projecao).limit(5):
    print(pedido["padronizado"])

Pedidos do Instagram: 478

{'cliente': 'Dra. Giovanna Pereira', 'cidade': 'Jesus do Sul', 'valor_total': 1791.72}
{'cliente': 'Léo Moura', 'cidade': 'Caldeira de Guerra', 'valor_total': 1840.74}
{'cliente': 'Benício Mendes', 'cidade': 'Lopes de Pires', 'valor_total': 1936.29}
{'cliente': 'Dante Farias', 'cidade': 'Souza de da Rocha', 'valor_total': 2166.33}
{'cliente': 'Kevin das Neves', 'cidade': 'Siqueira', 'valor_total': 15769.37}


In [63]:
#Consulta no formato original de um canal:
filtro = {"channel": "Mercado Livre", "shipping.state": "SP"}
print(f"Pedidos do Mercado Livre com entrega em SP: {pedidos_col.count_documents(filtro)}\n")

for pedido in pedidos_col.find(filtro, {"_id": 0, "ml_order_id": 1, "buyer.full_name": 1, "shipping": 1}).limit(5):
    print(pedido)

Pedidos do Mercado Livre com entrega em SP: 16

{'ml_order_id': 'ML5873270421', 'buyer': {'full_name': 'Matheus Vargas'}, 'shipping': {'city': 'Cardoso dos Dourados', 'state': 'SP', 'cost': 27.29}}
{'ml_order_id': 'ML3456062862', 'buyer': {'full_name': 'Thales Lima'}, 'shipping': {'city': 'Guerra do Norte', 'state': 'SP', 'cost': 33.32}}
{'ml_order_id': 'ML8893672237', 'buyer': {'full_name': 'Sr. Benício Cassiano'}, 'shipping': {'city': 'Vasconcelos do Sul', 'state': 'SP', 'cost': 45.05}}
{'ml_order_id': 'ML2780120449', 'buyer': {'full_name': 'Dr. Léo Ramos'}, 'shipping': {'city': 'Freitas', 'state': 'SP', 'cost': 26.28}}
{'ml_order_id': 'ML2857477095', 'buyer': {'full_name': 'Cecilia Casa Grande'}, 'shipping': {'city': 'Teixeira de Minas', 'state': 'SP', 'cost': 39.11}}


In [64]:
#Operador $gt
for pedido in (
    pedidos_col.find({"padronizado.valor_total": {"$gt": 10000}})
    .sort("padronizado.valor_total", -1)
    .limit(5)
):
    p = pedido["padronizado"]
    print(f"{moeda(p['valor_total']):>14} | {p['canal']:<13} | {p['cliente']}")

  R$ 54.795,42 | Facebook      | Isaac Camargo
  R$ 52.754,81 | Facebook      | Henry Gabriel Monteiro
  R$ 44.079,18 | Mercado Livre | Pedro Henrique Costa
  R$ 41.402,72 | Marketplace   | Guilherme Pinto
  R$ 39.643,08 | Marketplace   | Isabelly Andrade


In [65]:
# Operador $in
filtro = {"padronizado.forma_pagamento": {"$in": ["Pix", "Boleto"]}}
print(f"Pedidos pagos com Pix ou boleto: {pedidos_col.count_documents(filtro)}")

Pedidos pagos com Pix ou boleto: 976


In [66]:
#Intervalo de datas $gte / $lt
agora = datetime.now()
filtro = {"padronizado.data_pedido": {"$gte": agora - timedelta(days=30), "$lt": agora}}
print(f"Pedidos dos últimos 30 dias: {pedidos_col.count_documents(filtro)}")

Pedidos dos últimos 30 dias: 45


In [67]:
#Operador $or:
filtro = {
    "$or": [
        {"padronizado.canal": "Mercado Livre", "padronizado.valor_total": {"$gt": 5000}},
        {"padronizado.canal": "Marketplace", "padronizado.frete": 0},
    ]
}
print(f"Pedidos que atendem a pelo menos uma das condições: {pedidos_col.count_documents(filtro)}")

Pedidos que atendem a pelo menos uma das condições: 148


In [68]:
#Expressão regular ($regex):
filtro = {"padronizado.cliente": {"$regex": "^maria", "$options": "i"}}
print(f"Pedidos de clientes cujo nome começa com 'Maria': {pedidos_col.count_documents(filtro)}\n")

for pedido in pedidos_col.find(filtro, {"_id": 0, "padronizado.cliente": 1, "padronizado.canal": 1}).limit(5):
    print(pedido["padronizado"])

Pedidos de clientes cujo nome começa com 'Maria': 108

{'cliente': 'Maria Clara Moreira', 'canal': 'Mercado Livre'}
{'cliente': 'Mariana Rocha', 'canal': 'Mercado Livre'}
{'cliente': 'Maria Julia Moreira', 'canal': 'Marketplace'}
{'cliente': 'Mariane Barbosa', 'canal': 'Facebook'}
{'cliente': 'Maria Mendes', 'canal': 'Mercado Livre'}


In [69]:
#Resultado de uma consulta como DataFrame do pandas.
import pandas as pd

filtro = {"padronizado.canal": "Instagram"}
resultado = list(pedidos_col.find(filtro, {"_id": 0, "padronizado": 1}))
df = pd.json_normalize([r["padronizado"] for r in resultado])
df.head()

,canal,id_pedido_canal,cliente,telefone,cidade,estado,documento,tipo_documento,documento_valido,produtos,qtd_itens,valor_total,frete,forma_pagamento,data_pedido,alertas_qualidade
0,Instagram,IG-BB4897C5,Dra. Giovanna Pereira,03149414892,Jesus do Sul,None,None,None,None,[Ciranda Capitonê],3,1791.72,None,Cartao de debito,2023-05-16 10:12:35,[]
1,Instagram,IG-BB916505,Léo Moura,07165450155,Caldeira de Guerra,None,None,None,None,[Ciranda Elegance Big],2,1840.74,None,Pix,2024-01-07 19:00:52,[]
2,Instagram,IG-58B44226,Benício Mendes,09004421564,Lopes de Pires,None,None,None,None,[Ciranda Capitonê],3,1936.29,None,Cartao de credito,2026-06-13 15:20:28,[]
3,Instagram,IG-5D047329,Dante Farias,05122531875,Souza de da Rocha,None,None,None,None,[??rmario Expositor],3,2166.33,None,Cartao de debito,2025-12-08 09:58:16,[]
4,Instagram,IG-C75F59F3,Kevin das Neves,5508440675999,Siqueira,None,None,None,None,"[Lavatório Confort Cuba de Louça Branca, Kit E...",13,15769.37,None,Pix,2022-08-20 18:52:32,[]


In [70]:
#Leitura da segunda coleção:
total_logs = logs_col.count_documents({})
print(f"Documentos em {NOME_COLECAO_LOGS}: {total_logs}")

if total_logs == 0:
    print("Nenhum log ainda")
else:
    print("Últimos 5 eventos registrados:")
    for log in logs_col.find({}, {"_id": 0, "mensagem": 1}).sort("_id", -1).limit(5):
        print(f"  {log.get('mensagem')}")

Documentos em logs_tecnicos: 239
Últimos 5 eventos registrados:
  [2026-11-04 21:26:56]:REQUISICAO:GET /api/admin/kpis 200 1097.2ms
  [2026-11-04 21:26:55]:REQUISICAO:GET /api/admin/fornecedores 200 538.5ms
  [2026-11-04 21:26:55]:REQUISICAO:GET /api/admin/kpis 200 1406.1ms
  [2026-11-04 21:26:55]:REQUISICAO:GET /api/admin/clientes-por-regiao 200 383.8ms
  [2026-11-04 21:26:55]:REQUISICAO:GET /api/admin/fornecedores 200 625.2ms


In [71]:
# Proteção contra o limite de vazão do Cosmos DB, usada nas seções 9, 10 e 11.
def com_retentativa(operacao, max_tentativas=10):
    # Repete a operação recusada por limite de vazão (erro 16500), esperando cada vez mais.
    for tentativa in range(1, max_tentativas + 1):
        try:
            return operacao()
        except OperationFailure as erro:
            if erro.code != COD_LIMITE_VAZAO or tentativa == max_tentativas:
                raise
            time.sleep(min(0.5 * 2 ** tentativa, 30))


def em_lotes(colecao, metodo, filtro, *argumentos, tamanho_lote=TAMANHO_LOTE):
    """Aplica update_many ou delete_many aos documentos do filtro, em lotes. Devolve (encontrados, alterados).

    Os filtros usados aqui deixam de encontrar o documento depois de alterado ou removido: repetir um
    lote recusado no meio continua de onde parou, e a contagem final (pelo que sobrou) é exata.
    """
    ids = [doc["_id"] for doc in com_retentativa(lambda: list(colecao.find(filtro, {"_id": 1})))]
    for inicio in range(0, len(ids), tamanho_lote):
        lote = {"$and": [filtro, {"_id": {"$in": ids[inicio:inicio + tamanho_lote]}}]}
        com_retentativa(lambda: getattr(colecao, metodo)(lote, *argumentos))
    restantes = com_retentativa(lambda: colecao.count_documents(filtro)) if ids else 0
    return len(ids), len(ids) - restantes


print("Proteção contra o limite de vazão pronta.")

Proteção contra o limite de vazão pronta.


In [72]:
# update_many com $set: padroniza a escrita das formas de pagamento.
# Os canais mandam "Cartao de credito" e "Cartao de debito", sem acento; o bloco padronizado
# passa a usar a forma correta. O documento original do canal não é tocado.
CORRECOES_PAGAMENTO = {
    "Cartao de credito": "Cartão de crédito",
    "Cartao de debito": "Cartão de débito",
}

print("Antes:", sorted(pedidos_col.distinct("padronizado.forma_pagamento")))
for errado, certo in CORRECOES_PAGAMENTO.items():
    encontrados, alterados = em_lotes(
        pedidos_col, "update_many",
        {"padronizado.forma_pagamento": errado},
        {"$set": {"padronizado.forma_pagamento": certo}},
    )
    print(f"  '{errado}' -> '{certo}': {encontrados} encontrados, {alterados} alterados")
print("Depois:", sorted(pedidos_col.distinct("padronizado.forma_pagamento")))

Antes: ['Boleto', 'Cartao de credito', 'Cartao de debito', 'Pix']
  'Cartao de credito' -> 'Cartão de crédito': 530 encontrados, 530 alterados
  'Cartao de debito' -> 'Cartão de débito': 494 encontrados, 494 alterados
Depois: ['Boleto', 'Cartão de crédito', 'Cartão de débito', 'Pix']


In [75]:
# update_many com $set em um subdocumento: marca para conferência os pedidos que pedem
# um olhar humano antes do faturamento, valor acima do limite ou CPF/CNPJ que não passou na
# validação dos dígitos verificadores. O campo "revisao" não existia: o $set o cria.
# O filtro ignora quem já tem revisão, para não reabrir pedidos já conferidos.
FILTROS_REVISAO = {
    f"valor acima de {moeda(LIMITE_CONFERENCIA)}": {"padronizado.valor_total": {"$gt": LIMITE_CONFERENCIA}},
    "documento inválido": {"padronizado.documento_valido": False},
}


def marcar_para_revisao():
    marcados = {}
    for motivo, filtro in FILTROS_REVISAO.items():
        _, marcados[motivo] = em_lotes(
            pedidos_col, "update_many",
            {**filtro, "revisao.status": {"$exists": False}},
            {"$set": {"revisao.status": "pendente", "revisao.motivo": motivo}},
        )
    return marcados


for motivo, quantidade in marcar_para_revisao().items():
    print(f"Marcados para revisão ({motivo}): {quantidade}")
print(f"Total pendente de revisão: {pedidos_col.count_documents({'revisao.status': 'pendente'})}")

Marcados para revisão (valor acima de R$ 30.000,00): 8
Marcados para revisão (documento inválido): 0
Total pendente de revisão: 8


In [76]:
# update_one com $set e $unset: um pedido revisado sai da fila de pendentes.
# O status muda para "concluida", a data da revisão é gravada e o motivo é removido.
pedido = pedidos_col.find_one({"revisao.status": "pendente"}, {"padronizado.id_pedido_canal": 1, "revisao": 1})

if pedido is None:
    print("Nenhum pedido pendente de revisão.")
else:
    print("Antes: ", pedido["revisao"])
    resultado = com_retentativa(lambda: pedidos_col.update_one(
        {"_id": pedido["_id"]},
        {"$set": {"revisao.status": "concluida", "revisao.revisado_em": datetime.now(timezone.utc)},
         "$unset": {"revisao.motivo": ""}},
    ))
    depois = pedidos_col.find_one({"_id": pedido["_id"]}, {"revisao": 1})
    print("Depois:", depois["revisao"])
    print(f"Pedido {pedido['padronizado']['id_pedido_canal']}: {resultado.modified_count} documento alterado")

Antes:  {'status': 'pendente', 'motivo': 'valor acima de R$ 30.000,00'}
Depois: {'status': 'concluida', 'revisado_em': datetime.datetime(2026, 11, 17, 1, 45, 50, 163000)}
Pedido ML3418223376: 1 documento alterado


In [77]:
# delete_many: remove pedidos duplicados.
# O _id é o hash do documento inteiro, então o mesmo pedido (mesmo canal e mesmo número)
# enviado duas vezes com algum campo diferente entra como dois documentos. A agregação
# encontra esses casos; fica o primeiro que apareceu no arquivo e os demais são apagados.
pipeline = [
    {"$group": {
        "_id": {"canal": "$padronizado.canal", "id": "$padronizado.id_pedido_canal"},
        "documentos": {"$push": {"_id": "$_id", "linha": "$ingestao.linha"}},
        "quantidade": {"$sum": 1},
    }},
    {"$match": {"quantidade": {"$gt": 1}}},
]


def ids_duplicados(colecao):
    remover = []
    for grupo in colecao.aggregate(pipeline, allowDiskUse=True):
        documentos = sorted(grupo["documentos"], key=lambda d: d["linha"] or 0)
        remover.extend(d["_id"] for d in documentos[1:])  # mantém o primeiro
    return remover


remover = ids_duplicados(pedidos_col)
print(f"Pedidos na coleção:{pedidos_col.count_documents({})}")
print(f"Duplicados a remover:{len(remover)}")
if remover:
    _, removidos = em_lotes(pedidos_col, "delete_many", {"_id": {"$in": remover}})
    print(f"Removidos:{removidos}")
print(f"Pedidos únicos na coleção:{pedidos_col.count_documents({})}")

Pedidos na coleção:2000
Duplicados a remover:0
Pedidos únicos na coleção:2000


In [78]:
# delete_one: um pedido de teste é inserido e removido em seguida, sem tocar nos dados reais.
# 10.2 delete_one: um pedido de teste é inserido e removido em seguida, sem tocar nos dados reais.
pedido_teste = {
    "_id": "teste-delete-one",
    "padronizado": {"canal": "Teste", "id_pedido_canal": "TESTE-001", "valor_total": 0.0},
}
com_retentativa(lambda: pedidos_col.replace_one({"_id": pedido_teste["_id"]}, pedido_teste, upsert=True))

filtro_teste = {"padronizado.id_pedido_canal": "TESTE-001"}
print(f"Pedidos de teste antes:  {pedidos_col.count_documents(filtro_teste)}")
resultado = com_retentativa(lambda: pedidos_col.delete_one(filtro_teste))
print(f"Removidos:{resultado.deleted_count}")
print(f"Pedidos de teste depois: {pedidos_col.count_documents(filtro_teste)}")

Pedidos de teste antes:  1
Removidos:1
Pedidos de teste depois: 0


In [79]:
# delete_many na segunda coleção: retenção dos logs técnicos.
# Requisições com mais de DIAS_RETENCAO_LOGS dias são apagadas; os registros de erro são
# mantidos, porque servem para investigar falhas mesmo depois de muito tempo.
# O timestamp é gravado como texto ISO 8601, e nesse formato a comparação de textos ($lt)
# segue a ordem cronológica.
limite = (datetime.now() - timedelta(days=DIAS_RETENCAO_LOGS)).isoformat()
filtro_antigos = {"tipo_evento": "requisicao", "timestamp": {"$lt": limite}}

print(f"Logs técnicos na coleção:{logs_col.count_documents({})}")
print(f"Requisições anteriores a {limite[:10]}:{logs_col.count_documents(filtro_antigos)}")
_, removidas = em_lotes(logs_col, "delete_many", filtro_antigos)
print(f"Removidas:{removidas}")

Logs técnicos na coleção:239
Requisições anteriores a 2026-10-18:   57
Removidas:57


In [80]:
def rotina_qualidade():
    padronizadas = sum(
        em_lotes(pedidos_col, "update_many", {"padronizado.forma_pagamento": errado},
                 {"$set": {"padronizado.forma_pagamento": certo}})[1]
        for errado, certo in CORRECOES_PAGAMENTO.items()
    )
    marcados = sum(marcar_para_revisao().values())
    return {"formas_padronizadas": padronizadas, "marcados_para_revisao": marcados}


def rotina_deduplicacao():
    remover = ids_duplicados(pedidos_col)
    removidos = em_lotes(pedidos_col, "delete_many", {"_id": {"$in": remover}})[1] if remover else 0
    return {"duplicados_removidos": removidos}


def rotina_retencao_logs(dias=DIAS_RETENCAO_LOGS):
    limite = (datetime.now() - timedelta(days=dias)).isoformat()
    filtro = {"tipo_evento": "requisicao", "timestamp": {"$lt": limite}}
    return {"logs_removidos": em_lotes(logs_col, "delete_many", filtro)[1]}


def rotina_relatorio():
    resumo = {}
    for linha in pedidos_col.aggregate([
        {"$group": {"_id": "$padronizado.canal", "pedidos": {"$sum": 1},
                    "faturamento": {"$sum": "$padronizado.valor_total"}}},
    ]):
        resumo[linha["_id"]] = {"pedidos": linha["pedidos"], "faturamento": round(linha["faturamento"], 2)}
    return dict(sorted(resumo.items()))


ROTINAS = {
    "qualidade": rotina_qualidade,
    "deduplicacao": rotina_deduplicacao,
    "retencao_logs": rotina_retencao_logs,
    "relatorio": rotina_relatorio,
}


def executar_periodicamente(rotinas, vezes=REPETICOES_ROTINAS, intervalo_s=INTERVALO_ROTINAS_S):
    historico = []
    for execucao in range(1, vezes + 1):
        for nome, funcao in rotinas.items():
            inicio = time.time()
            resultado = funcao()
            historico.append({"execucao": execucao, "rotina": nome, "resultado": resultado})
            print(f"[{datetime.now():%H:%M:%S}] execução {execucao} | {nome}: {resultado} ({time.time() - inicio:.1f}s)")
        if execucao < vezes:
            time.sleep(intervalo_s)
    return historico


historico_rotinas = executar_periodicamente(ROTINAS)

[01:46:17] execução 1 | qualidade: {'formas_padronizadas': 0, 'marcados_para_revisao': 0} (0.6s)
[01:46:17] execução 1 | deduplicacao: {'duplicados_removidos': 0} (0.3s)
[01:46:17] execução 1 | retencao_logs: {'logs_removidos': 0} (0.1s)
[01:46:17] execução 1 | relatorio: {'Facebook': {'pedidos': 492, 'faturamento': 1812635.1}, 'Instagram': {'pedidos': 478, 'faturamento': 1591331.39}, 'Marketplace': {'pedidos': 508, 'faturamento': 2169898.66}, 'Mercado Livre': {'pedidos': 522, 'faturamento': 2077777.47}} (0.2s)


In [81]:
# Prova de que as rotinas podem ser repetidas: executadas de novo logo em seguida, as rotinas de
# atualização e remoção não encontram mais nada para mudar e devolvem zeros.
segunda_execucao = {nome: funcao() for nome, funcao in ROTINAS.items() if nome != "relatorio"}
print(segunda_execucao)
assert all(valor == 0 for resumo in segunda_execucao.values() for valor in resumo.values()), \
    "Alguma rotina alterou dados na segunda execução - ela não é segura para agendar."
print("Rotinas seguras para repetição: nenhuma alteração na segunda execução.")

{'qualidade': {'formas_padronizadas': 0, 'marcados_para_revisao': 0}, 'deduplicacao': {'duplicados_removidos': 0}, 'retencao_logs': {'logs_removidos': 0}}
Rotinas seguras para repetição: nenhuma alteração na segunda execução.


In [82]:
print("VERIFICAÇÃO FINAL")
print(f"Banco: {NOME_BANCO}")
print(f"Coleções: {sorted(db.list_collection_names())}")
print(f"Pedidos em {NOME_COLECAO_PEDIDOS}: {pedidos_col.count_documents({})}")
print(f"Arquivos lidos do Data Lake: {len(arquivos)}")

client.close()

VERIFICAÇÃO FINAL
Banco: conectabeauty
Coleções: ['logs_tecnicos', 'pedidos_canais_externos']
Pedidos em pedidos_canais_externos: 2000
Arquivos lidos do Data Lake: 1
